## 将生僻字JSONL数据转换为 LangChain Document

In [22]:
import json
from langchain_core.documents import Document

chunks_file_path = r"knowledgeBase\pdfParse\cleaned_data\rare_hanzi_integrated.jsonl"

langch_docs = []
with open(chunks_file_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        
        # 提取需要拼接的字段
        hanzi = obj.get("汉字", "")
        duyin = obj.get("读音", "")
        chaizi = obj.get("拆字", [])
        jieshi = obj.get("解释", "")
        chuxian_shuyu = obj.get("出现在术语", [])
        
        # 将拆字列表转换为字符串
        chaizi_str = "、".join(chaizi) if chaizi else ""
        
        # 将出现在术语列表转换为字符串
        chuxian_shuyu_str = "、".join(chuxian_shuyu) if chuxian_shuyu else ""
        
        # 拼接为连贯句子
        page_content_parts = []
        if hanzi:
            page_content_parts.append(f"'{hanzi}'字")
        if duyin:
            page_content_parts.append(f"其为读音：{duyin}")
        if chaizi_str:
             page_content_parts.append(f"可以拆字表示为：{chaizi_str}")
        if jieshi:
            page_content_parts.append(f"意思是：{jieshi}")
        if chuxian_shuyu_str:
            page_content_parts.append(f"出现在术语：{chuxian_shuyu_str}")
        
        # 用分隔符连接各部分
        page_content = "；".join(page_content_parts)
        
        # 构建 metadata（排除已用于 page_content 的字段）
        metadata = {}
        for key, value in obj.items():
            if key not in ["汉字", "读音", "拆字", "解释","出现在术语"]:
                metadata[key] = value
        
        langch_docs.append(
            Document(
                page_content=page_content,
                metadata=metadata
            )
        )

In [20]:
import pprint
print("该库包含documents数量：" + str(len(langch_docs)))
pprint.pprint(f"{langch_docs[11].page_content}")
print("格式化展示该条metadata：" + pprint.pformat(langch_docs[11].metadata))
print(langch_docs[11].metadata.get("UNICODE"))

该库包含documents数量：126
("'笏'字；其为读音：hù；可以拆字表示为：𥫗（竹）、勿；意思是：古代大臣上朝拿着的手板，用玉、象牙或竹片制成，上面可以记事。例如朝（ cháo "
 '）～。“京兆尹郑叔则，怫然曳～却立”。；出现在术语：笏头碣、笏首')
格式化展示该条metadata：{'UNICODE': 'U+7B0F',
 '字形相似汉字': ['筛', '筋', '筒', '箫', '篱', '筏', '简', '第', '箭', '笋'],
 '数据来源': '汉语字典 https://www.hanyuguoxue.com/zidian/zi-31503',
 '替代字': None,
 '涉及术语': ['笏头碣', '笏首']}
U+7B0F


### 使用阿里云模型，向量化、归一化

In [23]:
import os
import numpy as np
from openai import OpenAI
from langchain_core.embeddings import Embeddings

# 设置阿里云API密钥
API_KEY = os.getenv("DASHSCOPE_API_KEY")
if not API_KEY:
    raise ValueError("请设置环境变量 DASHSCOPE_API_KEY")
BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "text-embedding-v4"

# 初始化OpenAI客户端
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

class AliyunEmbeddings(Embeddings):
    """阿里云文本嵌入模型，返回L2归一化后的向量（便于余弦相似度计算）"""
    def __init__(self, client, model_name="text-embedding-v4", batch_size=10):
        self.client = client
        self.model_name = model_name
        self.batch_size = batch_size

    def _normalize(self, vec):
        """L2归一化"""
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def embed_documents(self, texts):
        all_embeddings = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i+self.batch_size]
            resp = self.client.embeddings.create(model=self.model_name, input=batch)
            batch_embeddings = [self._normalize(np.array(item.embedding)) for item in resp.data]
            all_embeddings.extend(batch_embeddings)
        return all_embeddings

    def embed_query(self, text):
        resp = self.client.embeddings.create(model=self.model_name, input=text)
        vec = np.array(resp.data[0].embedding)
        return self._normalize(vec)

# 实例化嵌入模型（构建和检索均使用此实例）
aliyun_emb = AliyunEmbeddings(client, model_name=MODEL_NAME, batch_size=10)

### 使用FAISS，向量本地存储

In [ ]:
from langchain_community.vectorstores import FAISS
from tqdm import tqdm  # 需要先安装：pip install tqdm

# 提取文本、元数据和 ID
texts = [doc.page_content for doc in langch_docs]
metadatas = [doc.metadata for doc in langch_docs]
doc_ids = [doc.metadata.get('UNICODE') for doc in langch_docs]

# 分批生成嵌入，并显示进度
batch_size = aliyun_emb.batch_size
embeddings = []

for i in tqdm(range(0, len(texts), batch_size), desc="生成嵌入进度"):
    batch_texts = texts[i:i+batch_size]
    batch_emb = aliyun_emb.embed_documents(batch_texts)  # 返回归一化的 numpy 数组列表
    embeddings.extend(batch_emb)

# 使用预生成的嵌入构建 FAISS 向量库
# 默认创建的 FAISS 索引类型为 IndexFlatL2。这是一种基于欧几里得距离（L2）的精确搜索（Flat）索引，适用于中小规模数据集的精准相似性检索
vectorstore = FAISS.from_embeddings(
    text_embeddings=list(zip(texts, embeddings)),  # (文本, 嵌入向量) 对
    embedding=aliyun_emb,                           # 仍需传入 embedding 对象用于查询
    metadatas=metadatas,
    ids=doc_ids
)

# 保存到本地
folder_path = "knowledgeBase\\chunks\\hanzi_faiss_index"  # 注意 Windows 路径转义或使用正斜杠
vectorstore.save_local(folder_path)
print(f"--- 向量库已成功保存至目录: {folder_path} ---")

生成嵌入进度: 100%|██████████| 13/13 [00:05<00:00,  2.34it/s]


--- 向量库已成功保存至目录: knowledgeBase\chunks\hanzi_faiss_index ---


### 本地FAISS向量库加载和查找

In [27]:

from langchain_community.vectorstores import FAISS

index_folder = r"knowledgeBase\chunks\hanzi_faiss_index"       

local_db = FAISS.load_local(
    folder_path=index_folder, 
    embeddings=aliyun_emb,
    index_name="index",#默认为 "index"。它会去文件夹里找 index.faiss 和 index.pkl 两个文件
    allow_dangerous_deserialization=True
) 

### 根据unicode查找对应的汉字信息，支持单条更新
target_unicode = "U+7B0F"

# 直接从 docstore 中检索
if target_unicode in local_db.docstore._dict:
    doc = local_db.docstore.search(target_unicode)
    print(f"找到汉字: {doc.page_content[:30]}...")
    print(f"元数据: {doc.metadata}")

    # 1. 找到该 Unicode 在 FAISS 内部的数字行号
    # 翻转映射表：从 ID 找 行号
    id_to_index = {v: k for k, v in local_db.index_to_docstore_id.items()}
    row_idx = id_to_index.get(target_unicode)

    if row_idx is not None:
        # 2. 直接从 FAISS 矩阵中提取原始向量
        original_vector = local_db.index.reconstruct(row_idx)
        print(f"数据库中存储的原始向量：{original_vector[:10]}")
        print(f"向量维度：{len(original_vector)}")  
else:
    print("未找到该 Unicode 对应的记录")

找到汉字: '笏'字；其为读音：hù；可以拆字表示为：𥫗（竹）、勿；意思...
元数据: {'UNICODE': 'U+7B0F', '替代字': None, '字形相似汉字': ['筛', '筋', '筒', '箫', '篱', '筏', '简', '第', '箭', '笋'], '数据来源': '汉语字典 https://www.hanyuguoxue.com/zidian/zi-31503', '涉及术语': ['笏头碣', '笏首']}
数据库中存储的原始向量：[ 0.03413424 -0.07694513  0.0292988   0.03181443  0.04401599 -0.00707615
 -0.02655722  0.08242831 -0.01958275  0.01046171]
向量维度：1024


### 从已保存的FAISS索引加载数据，构建倒排索引和文档向量数组


In [28]:
import os
import pickle
import numpy as np
import jieba
from langchain_community.vectorstores import FAISS

# 路径设置
base_path = "knowledgeBase/chunks"
faiss_index_path = os.path.join(base_path, "hanzi_faiss_index")
vectors_path = os.path.join(base_path, "vectors.npy")
doc_ids_path = os.path.join(base_path, "doc_ids.npy")
inverted_index_path = os.path.join(base_path, "inverted_index.pkl")
doc_lengths_path = os.path.join(base_path, "doc_lengths.pkl")
stats_path = os.path.join(base_path, "stats.pkl")

# 1. 加载FAISS向量库（使用之前保存的索引）
print("正在加载FAISS索引...")
vectorstore = FAISS.load_local(faiss_index_path, aliyun_emb, allow_dangerous_deserialization=True)

# 2. 提取所有文档向量（从FAISS索引中重建）
index = vectorstore.index
num_vectors = index.ntotal
print(f"索引中包含 {num_vectors} 个向量。")

# FAISS IndexFlatL2 或 IndexFlatIP 均支持 reconstruct_n
vectors = index.reconstruct_n(0, num_vectors)  #reconstruct_n 方法，从索引中重建所有向量。参数 0 表示从第 0 个向量开始，num_vectors 表示重建的数量。
#返回一个形状为 (num_vectors, dim) 的 NumPy 数组
norms = np.linalg.norm(vectors, axis=1, keepdims=True)#计算每个向量的 L2 范数（欧几里得长度），axis=1 表示对每个向量计算，keepdims=True 保持维度以便广播
vectors = vectors / np.where(norms > 0, norms, 1)  # 归一化，防止除零

# 3. 获取文档ID列表和文档内容
# index_to_docstore_id 是 FAISS 内部映射：index_id -> docstore_id
# docstore 是字典：docstore_id -> Document
index_to_doc_id = vectorstore.index_to_docstore_id  # dict {idx: doc_id}
docstore = vectorstore.docstore  # InMemoryDocstore 对象，可以像字典一样操作

doc_ids = []          # 按索引顺序的文档ID列表
langch_docs = []      # 按索引顺序的 Document 对象列表
for idx in range(num_vectors):
    doc_id = index_to_doc_id[idx]
    doc_ids.append(doc_id)
    doc = docstore.search(doc_id)  # 通过 docstore 获取 Document
    langch_docs.append(doc)

# 4. 保存向量数组和文档ID（供后续检索使用）
np.save(vectors_path, vectors)
np.save(doc_ids_path, np.array(doc_ids))
print(f"向量已保存至 {vectors_path}，文档ID已保存至 {doc_ids_path}")

# 5. 构建倒排索引
print("正在构建倒排索引...")
inverted_index = {}        # 初始化 {词: {doc_id: 词频}}字典
doc_lengths = {}           # 舒适化 {doc_id: 文档词数}字典
N = len(langch_docs)

for i, doc in enumerate(langch_docs):
    doc_id = doc_ids[i]
    # 结巴分词
    words = jieba.lcut(doc.page_content)
    length = len(words)
    doc_lengths[doc_id] = length
    
    # 统计词频
    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    
    # 更新倒排索引
    for w, f in freq.items():
        if w not in inverted_index:
            inverted_index[w] = {}
        inverted_index[w][doc_id] = f

# 计算平均文档长度
avg_len = sum(doc_lengths.values()) / N

# 保存倒排索引、文档长度和统计信息
with open(inverted_index_path, 'wb') as f:
    pickle.dump(inverted_index, f)
with open(doc_lengths_path, 'wb') as f:
    pickle.dump(doc_lengths, f)
with open(stats_path, 'wb') as f:
    pickle.dump({'N': N, 'avg_len': avg_len}, f)

print(f"倒排索引构建完成，文档总数：{N}，平均长度：{avg_len:.2f}")

正在加载FAISS索引...
索引中包含 126 个向量。
向量已保存至 knowledgeBase/chunks\vectors.npy，文档ID已保存至 knowledgeBase/chunks\doc_ids.npy
正在构建倒排索引...


Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\junqiang\AppData\Local\Temp\jieba.cache
Loading model cost 0.361 seconds.
Prefix dict has been built successfully.


倒排索引构建完成，文档总数：126，平均长度：58.25


### 加载检索所需的数据

In [29]:
# 加载FAISS向量库（使用之前保存的索引）
vectorstore = FAISS.load_local(faiss_index_path, aliyun_emb, allow_dangerous_deserialization=True)

# 加载向量数组和文档ID
doc_embeddings = np.load(vectors_path)
doc_ids = np.load(doc_ids_path, allow_pickle=True).tolist()

# 建立doc_id到索引的映射，便于快速获取向量
doc_id_to_index = {doc_id: idx for idx, doc_id in enumerate(doc_ids)}

# 加载倒排索引相关数据
with open(inverted_index_path, 'rb') as f:
    inverted_index = pickle.load(f)
with open(doc_lengths_path, 'rb') as f:
    doc_lengths = pickle.load(f)
with open(stats_path, 'rb') as f:
    stats = pickle.load(f)
N = stats['N']
avg_len = stats['avg_len']

print("数据加载完成。")

数据加载完成。


### 定义检索函数（两阶段：向量召回 + BM25重排序）


In [34]:
import math
import numpy as np

def bm25_score(query_words, doc_id, inverted_index, doc_lengths, N, avg_len, k1=1.5, b=0.75):
    """
    计算单个文档的BM25得分（标准实现）
    :param query_words: 查询分词列表
    :param doc_id: 文档ID
    :param inverted_index: 倒排索引 {词: {doc_id: 词频}}
    :param doc_lengths: 文档长度 {doc_id: 词数}
    :param N: 文档总数
    :param avg_len: 平均文档长度
    :param k1, b: BM25参数
    :return: BM25得分
    """
    score = 0.0
    doc_len = doc_lengths[doc_id]
    for w in query_words:
        if w not in inverted_index or doc_id not in inverted_index[w]:
            continue
        freq = inverted_index[w][doc_id]
        # 逆文档频率 (IDF)
        n_w = len(inverted_index[w])  # 包含词w的文档数
        idf = math.log((N - n_w + 0.5) / (n_w + 0.5) + 1)
        # 词频因子 (TF)
        tf_part = freq * (k1 + 1) / (freq + k1 * (1 - b + b * (doc_len / avg_len)))
        score += idf * tf_part
    return score

def hybrid_search(query, k_vector=8, k_final=3, alpha=0.5):
    """
    两阶段混合检索：
    1. 向量召回：使用FAISS（L2距离）获取 Top-(k_vector*2) 候选文档
    2. 精确计算余弦相似度（基于保存的归一化向量）和BM25得分，加权融合后取Top-k_final
    :param query: 用户查询字符串
    :param k_vector: 第一阶段向量召回的数量（实际多取一倍以确保覆盖）
    :param k_final: 最终返回的文档数
    :param alpha: 余弦相似度的权重，BM25权重为 1-alpha
    :return: 列表，每个元素为 (Document对象, 综合得分)
    """
    # 1. 查询向量化（已归一化）
    query_vec = aliyun_emb.embed_query(query)  # 返回归一化向量

    # 2. 使用FAISS进行第一阶段召回（基于L2距离）
    #   注意：vectorstore 是基于L2距离的IndexFlatL2，返回的score是L2距离（越小越相似）
    #   因为向量已归一化，L2距离与余弦相似度单调负相关，所以用L2召回不会漏掉最相似文档
    docs_and_scores = vectorstore.similarity_search_with_score_by_vector(
        query_vec, k=k_vector * 2
    )
    candidate_docs = [doc for doc, _ in docs_and_scores]  # 候选文档列表

    # 3. 提取候选文档ID并计算精确的余弦相似度（利用保存的向量数组）
    cos_scores = []
    for doc in candidate_docs:
        doc_id = doc.metadata['UNICODE']
        idx = doc_id_to_index[doc_id]          # 文档ID在向量数组中的索引
        doc_vec = doc_embeddings[idx]          # 已归一化
        cos_sim = np.dot(query_vec, doc_vec)   # 归一化后点积即余弦相似度
        cos_scores.append(cos_sim)

    # 4. 对候选文档计算BM25得分
    query_words = list(jieba.cut(query))
    bm25_scores = []
    for doc in candidate_docs:
        doc_id = doc.metadata['UNICODE']
        bm25 = bm25_score(query_words, doc_id, inverted_index, doc_lengths, N, avg_len)
        bm25_scores.append(bm25)

    # 5. 得分融合
    cos_scores = np.array(cos_scores)
    bm25_scores = np.array(bm25_scores)

    # 将BM25得分归一化到[0,1]区间
    bm25_min, bm25_max = bm25_scores.min(), bm25_scores.max()
    if bm25_max - bm25_min > 1e-9:
        bm25_norm = (bm25_scores - bm25_min) / (bm25_max - bm25_min)
    else:
        bm25_norm = np.zeros_like(bm25_scores)

    # 余弦相似度截断到[0,1]（理论上归一化向量点积在[-1,1]，但检索场景多为正，保留非负）
    cos_scores = np.clip(cos_scores, 0, 1)

    # 加权融合
    hybrid_scores = alpha * cos_scores + (1 - alpha) * bm25_norm

    # 6. 按综合得分降序排序，取前k_final
    sorted_indices = np.argsort(hybrid_scores)[::-1]
    results = []
    for idx in sorted_indices[:k_final]:
        doc = candidate_docs[idx]
        score = hybrid_scores[idx]
        results.append((doc, score))

    return results

In [40]:
# 示例使用
if __name__ == "__main__":
    query = "拆字为木、宛的字是什么？"
    top_docs = hybrid_search(query)
    print(f"查询: {query}\n")
    for i, (doc, score) in enumerate(top_docs):
        print(f"Top {i+1} (得分: {score:.4f}):")
        print(f"内容: {doc.page_content[:200]}...")
        print(f"元数据: {doc.metadata}\n")

查询: 拆字为木、宛的字是什么？

Top 1 (得分: 0.8636):
内容: '椀'字；其为读音：wǎn；可以拆字表示为：木、宛；意思是：同“碗”。...
元数据: {'UNICODE': 'U+6900', '替代字': None, '字形相似汉字': ['榨', '植', '棍', '棕', '棺', '榄', '槐', '概', '械', '梭'], '数据来源': '汉语字典 https://www.hanyuguoxue.com/zidian/zi-26880', '涉及术语': None}

Top 2 (得分: 0.3704):
内容: '楅'字；其为读音：bī；可以拆字表示为：木、畐；意思是：①拴在牛角上防止牛顶人的横木：“凡祭祀，饰其牛牲，设其～衡。” ②古代行乡射礼时插箭的器具：“命弟子设～。” ③木门后用以连结门板的横衬。；出现在术语：楅、罗文楅、承拐楅...
元数据: {'UNICODE': 'U+6945', '替代字': None, '字形相似汉字': ['梧', '栖', '椿', '楷', '概', '榴', '榕', '槽', '梗', '橱'], '数据来源': '汉语字典 https://www.hanyuguoxue.com/zidian/zi-26949', '涉及术语': ['楅', '罗文楅', '承拐楅']}

Top 3 (得分: 0.3632):
内容: '榫'字；其为读音：sǔn；可以拆字表示为：木、隼；意思是：器物两部分利用凹凸相接的凸出的部分。例如～子。～卯。；出现在术语：透榫、箍头榫、半榫、燕尾榫...
元数据: {'UNICODE': 'U+69AB', '替代字': None, '字形相似汉字': ['樟', '橱', '橄', '校', '椎', '榜', '樱', '棱', '檀', '椒'], '数据来源': '汉语字典 https://www.hanyuguoxue.com/zidian/zi-27051', '涉及术语': ['透榫', '箍头榫', '半榫', '燕尾榫']}

